# Milestone 1 Exploration: Amazon Reviews 2023

This notebook prepares the retrieval corpus for Milestone 1 of the Amazon product query assistant project.

## Goal
The goal of this notebook is to:
- load one Amazon Reviews 2023 category
- inspect the review and metadata files
- choose useful fields for retrieval
- build a clean corpus for downstream BM25 and semantic search

## Chosen category
We use the `Video_Games` category.

## Retrieval design choice

We use **one row per review**, then enrich each review with product metadata.

This is a good fit for Milestone 1 because:
- the app needs to show review text and rating
- BM25 and semantic retrieval should use the same document unit
- metadata such as title, categories, features, and description can improve matching

In [ ]:
from pathlib import Path
import pandas as pd

from src.utils import load_jsonl_gz, build_corpus, save_corpus

## Load raw files

The project expects these files in `data/raw/`:
- `Video_Games.jsonl.gz`
- `meta_Video_Games.jsonl.gz`

In [ ]:
reviews_path = Path("../data/raw/Video_Games.jsonl.gz")
meta_path = Path("../data/raw/meta_Video_Games.jsonl.gz")

reviews_df = load_jsonl_gz(reviews_path)
meta_df = load_jsonl_gz(meta_path)

print("Reviews shape:", reviews_df.shape)
print("Metadata shape:", meta_df.shape)

## Sample review records

In [ ]:
reviews_df.head()

## Sample metadata records

In [ ]:
meta_df.head()

## Inspect available columns

In [ ]:
print("Review columns:")
print(reviews_df.columns.tolist())

print("\nMetadata columns:")
print(meta_df.columns.tolist())

## Missingness in key fields

We check the main fields that are likely to be useful for retrieval and display.

In [ ]:
key_cols_reviews = [col for col in ["parent_asin", "asin", "title", "text", "rating"] if col in reviews_df.columns]
key_cols_meta = [col for col in ["parent_asin", "asin", "title", "description", "features", "categories", "price"] if col in meta_df.columns]

print("Missing values in review fields:")
print(reviews_df[key_cols_reviews].isna().sum())

print("\nMissing values in metadata fields:")
print(meta_df[key_cols_meta].isna().sum())

## Chosen retrieval fields

We combine the following fields into one `retrieval_text` column:

From reviews:
- review title
- review text
- rating is kept as metadata for display

From product metadata:
- product title
- description
- features
- categories
- price is kept as metadata for later use if needed

This gives the retrievers both product-level and review-level context.

## Preprocessing decisions

For Milestone 1, we use simple and reproducible preprocessing:

- lowercase text
- remove most punctuation
- normalize whitespace
- use whitespace tokenization for BM25
- keep the semantic retriever on the same cleaned retrieval text

This keeps the pipeline simple while still making BM25 and semantic search comparable.

## Build the retrieval corpus

In [ ]:
corpus_df = build_corpus(reviews_df, meta_df)
print("Corpus shape:", corpus_df.shape)
corpus_df.head()

## Inspect the final retrieval text

In [ ]:
preview_cols = [col for col in ["product_title", "review_title", "text", "retrieval_text"] if col in corpus_df.columns]
corpus_df[preview_cols].head(3)

## Save processed corpus

This file will be used later by:
- `src/bm25.py`
- `src/semantic.py`
- the app layer

In [ ]:
save_corpus(corpus_df, "../data/processed/video_games_corpus.parquet")
print("Saved corpus to ../data/processed/")

## Summary

This notebook completes the data exploration and corpus preparation part for Milestone 1.

After this notebook, the next steps are:
1. build the BM25 index
2. build the semantic embedding + FAISS index
3. persist both indexes locally for reuse